``@tf.function`` :
- Python 本身函数的执行效率不高，如果 Python 函数能够像 tf 的库函数来使用是不是很好？因此 tf 针对编译器和各种设备做了各种优化
- 将 python 函数编译成 tensorflow 的图（把 python 原生代码变为图，提高运行效率）
- 易于将模型导出成为 GraphDef + checkpoint 或者 SavedModel
- 使得 eager execution 可以默认打开（有 ``tf.function`` 可以保存图结构）
- **坏处是可能会影响调试**

``tf.py_function`` :
- 把普通的 Python 代码比如 while, for, if 等转成 tensorflow 中的 op
- 不会把 Python 函数内部的操作转变成图固化下来，**可以正常调试**

In [1]:
!python3 --version

Python 3.11.11


In [2]:
!nvidia-smi

Sun Jun  8 12:56:40 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       1MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
import os
import sys
import time
import tensorflow as tf

from tensorflow import keras

print(tf.__version__)
print(sys.version_info)
for module in mpl, np, pd, sklearn, tf, keras:
    print(module.__name__, module.__version__)

2025-06-08 12:56:45.054349: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749387405.492116      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749387405.616179      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


2.18.0
sys.version_info(major=3, minor=11, micro=11, releaselevel='final', serial=0)
matplotlib 3.7.2
numpy 1.26.4
pandas 2.2.3
sklearn 1.2.2
tensorflow 2.18.0
keras._tf_keras.keras 3.8.0


In [4]:
import tensorflow as tf

print(f"TensorFlow Version: {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
print(f"Num GPUs Available: {len(gpus)}")

if gpus:
    print(f"GPUs available: {gpus}")
    try:
        # 打印每个GPU的详细信息
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True) # 推荐设置，按需分配显存
            print(f"Details for {gpu.name}:")
            # 可以尝试执行一个小操作来确认
        print("GPU is available and TensorFlow can see it!")

        # 尝试一个简单的GPU运算
        print("\nAttempting a simple computation on GPU...")
        with tf.device('/GPU:0'): # 明确指定在第一个GPU上运行
            a = tf.constant([[1.0, 2.0], [3.0, 4.0]])
            b = tf.constant([[1.0, 2.0], [3.0, 4.0]])
            c = tf.matmul(a, b)
        print("Matrix multiplication result from GPU:")
        print(c.numpy())
        print("If no errors occurred, GPU is working!")

    except RuntimeError as e:
        print(f"RuntimeError during GPU setup or test: {e}")
else:
    print("GPU not available to TensorFlow. TensorFlow will run on CPU.")

TensorFlow Version: 2.18.0
Num GPUs Available: 2
GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
Details for /physical_device:GPU:0:
Details for /physical_device:GPU:1:
GPU is available and TensorFlow can see it!

Attempting a simple computation on GPU...
Matrix multiplication result from GPU:
[[ 7. 10.]
 [15. 22.]]
If no errors occurred, GPU is working!


I0000 00:00:1749387423.849670      35 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1749387423.850430      35 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [5]:
# tf.function and auto-graph.
# 自己实现 elu 激活函数,如果 scale 不为 1，那就是 selu
def scaled_elu(z, scale=1.0, alpha=1.0):
    # z >= 0 ? scale * z : scale * alpha * tf.nn.elu(z)
    is_positive = tf.greater_equal(z, 0.0)
#     return scale * tf.where(is_positive, z, alpha * tf.nn.elu(z))
    return scale * tf.where(is_positive, z, alpha * (tf.math.exp(z)-1))

# 运行一下，这还是py函数
print(scaled_elu(tf.constant(-3.)))
print(scaled_elu(tf.constant([-3., -2.5])))

tf.Tensor(-0.95021296, shape=(), dtype=float32)
tf.Tensor([-0.95021296 -0.917915  ], shape=(2,), dtype=float32)


In [6]:
#把py实现的函数变为图实现的函数
#scaled_elu_tf就是图
scaled_elu_tf = tf.function(scaled_elu)  # 注意这里不能加括号
print(scaled_elu_tf(tf.constant(-3.)))
print(scaled_elu_tf(tf.constant([-3., -2.5])))

tf.Tensor(-0.95021296, shape=(), dtype=float32)
tf.Tensor([-0.95021296 -0.917915  ], shape=(2,), dtype=float32)


In [7]:
#可以通过这种方式找回原来的py函数
print(scaled_elu_tf.python_function is scaled_elu)
print(scaled_elu)
print(scaled_elu_tf)#tf的函数的执行效率比较高

True
<function scaled_elu at 0x7fcef9c9b7e0>


In [8]:
#我们来测试一下性能，100万个数
%timeit scaled_elu(tf.random.normal((1000, 1000)))
%timeit scaled_elu_tf(tf.random.normal((1000, 1000)))
# 以下结果为运行误差内 -_-#

878 µs ± 14.3 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)
968 µs ± 14.9 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [9]:
# 1 + 1/2 + 1/2^2 + ... + 1/2^n
#加了@tf.function装饰后就变为图结果，但是输入类型上不会有变化
@tf.function
def converge_to_2(n_iters):
    total = tf.constant(0.)
    increment = tf.constant(1.)
    for _ in range(n_iters):
        total += increment
        increment /= 2.0
    return total
print(converge_to_2)
print(converge_to_2(20))

tf.Tensor(1.9999981, shape=(), dtype=float32)


In [10]:
#如何看tf的图的代码
def display_tf_code(func):
    code = tf.autograph.to_code(func)
    from IPython.display import display, Markdown
    display(Markdown('```python\n{}\n```'.format(code)))

In [11]:
#传普通py函数,返回的是tf图的代码
display_tf_code(scaled_elu)

```python
def tf__scaled_elu(z, scale=None, alpha=None):
    with ag__.FunctionScope('scaled_elu', 'fscope', ag__.ConversionOptions(recursive=True, user_requested=True, optional_features=(), internal_convert_user_code=True)) as fscope:
        do_return = False
        retval_ = ag__.UndefinedReturnValue()
        is_positive = ag__.converted_call(ag__.ld(tf).greater_equal, (ag__.ld(z), 0.0), None, fscope)
        try:
            do_return = True
            retval_ = ag__.ld(scale) * ag__.converted_call(ag__.ld(tf).where, (ag__.ld(is_positive), ag__.ld(z), ag__.ld(alpha) * (ag__.converted_call(ag__.ld(tf).math.exp, (ag__.ld(z),), None, fscope) - 1)), None, fscope)
        except:
            do_return = False
            raise
        return fscope.ret(retval_, do_return)

```

In [12]:
# 这个的前提是去除converge_to_2的装饰
# 因为converge_to_2有@tf.function标注，去掉应该就没问题了。to_code函数的输入是module,
# class, method, function, traceback, frame, or code object。不能是tf function.
display_tf_code(converge_to_2.python_function)

```python
def tf__converge_to(n_iters):
    with ag__.FunctionScope('converge_to_2', 'fscope', ag__.ConversionOptions(recursive=True, user_requested=True, optional_features=(), internal_convert_user_code=True)) as fscope:
        do_return = False
        retval_ = ag__.UndefinedReturnValue()
        total = ag__.converted_call(ag__.ld(tf).constant, (0.0,), None, fscope)
        increment = ag__.converted_call(ag__.ld(tf).constant, (1.0,), None, fscope)

        def get_state():
            return (total, increment)

        def set_state(vars_):
            nonlocal increment, total
            total, increment = vars_

        def loop_body(itr):
            nonlocal increment, total
            _ = itr
            total = ag__.ld(total)
            total += increment
            increment = ag__.ld(increment)
            increment /= 2.0
        _ = ag__.Undefined('_')
        ag__.for_stmt(ag__.converted_call(ag__.ld(range), (ag__.ld(n_iters),), None, fscope), None, loop_body, get_state, set_state, ('total', 'increment'), {'iterate_names': '_'})
        try:
            do_return = True
            retval_ = ag__.ld(total)
        except:
            do_return = False
            raise
        return fscope.ret(retval_, do_return)

```

In [13]:
# tf要把变量定义在函数外面，不能放里边，避免在函数内重复地申请空间
var = tf.Variable(0.)

@tf.function
def add_21():
    return var.assign_add(21) # +=

print(add_21())

tf.Tensor(21.0, shape=(), dtype=float32)


In [14]:
# cube计算立方，py是泛型设计，我们通过input_signature加类型限制可以防止调错
# @tf.function(input_signature=[tf.TensorSpec([None], tf.int32, name='x')])
@tf.function
def cube(z):
    return tf.pow(z, 3)

try:
    print(cube(tf.constant([1., 2., 3.])))
except ValueError as ex:
    print(ex)

print('-'*50)
# 这行没问题
print(cube(tf.constant([1, 2, 3])))
print(cube)

tf.Tensor([ 1.  8. 27.], shape=(3,), dtype=float32)
--------------------------------------------------
tf.Tensor([ 1  8 27], shape=(3,), dtype=int32)


In [15]:
# @tf.function py func -> tf graph
# get_concrete_function -> 给tf.function add input signature -> SavedModel

# 加限制，变成另外一个图
cube_func_int32 = cube.get_concrete_function(
    tf.TensorSpec([None], tf.int32))
print(cube_func_int32)

print('---------------')

print(cube)

print('----')

try:
    print(cube_func_int32(tf.constant([1, 2, 3])))
except Exception as ex:
    print(ex)

ConcreteFunction Input Parameters:
  z (POSITIONAL_OR_KEYWORD): TensorSpec(shape=(None,), dtype=tf.int32, name=None)
Output Type:
  TensorSpec(shape=(None,), dtype=tf.int32, name=None)
Captures:
  None
---------------
----
tf.Tensor([ 1  8 27], shape=(3,), dtype=int32)


In [16]:
#我们只要看原来函数和新生成的是否一致
# print(cube_func_int32 is cube.get_concrete_function())
print(cube.get_concrete_function(
    tf.constant([1, 2, 3])))
print('----')
print(cube_func_int32)
print('----')
print(cube_func_int32 is cube.get_concrete_function(
    tf.constant([1, 2, 3])))

ConcreteFunction Input Parameters:
  z (POSITIONAL_OR_KEYWORD): TensorSpec(shape=(3,), dtype=tf.int32, name=None)
Output Type:
  TensorSpec(shape=(3,), dtype=tf.int32, name=None)
Captures:
  None
----
ConcreteFunction Input Parameters:
  z (POSITIONAL_OR_KEYWORD): TensorSpec(shape=(None,), dtype=tf.int32, name=None)
Output Type:
  TensorSpec(shape=(None,), dtype=tf.int32, name=None)
Captures:
  None
----
False


In [17]:
print(cube_func_int32)
cube_func_int32.graph

ConcreteFunction Input Parameters:
  z (POSITIONAL_OR_KEYWORD): TensorSpec(shape=(None,), dtype=tf.int32, name=None)
Output Type:
  TensorSpec(shape=(None,), dtype=tf.int32, name=None)
Captures:
  None


In [18]:
#看下图定义都有哪些操作,了解即可
cube_func_int32.graph.get_operations()

[<tf.Operation 'z' type=Placeholder>,
 <tf.Operation 'Pow/y' type=Const>,
 <tf.Operation 'Pow' type=Pow>,
 <tf.Operation 'Identity' type=Identity>]

In [19]:
pow_op = cube_func_int32.graph.get_operations()[0]
print(pow_op)

name: "z"
op: "Placeholder"
attr {
  key: "_user_specified_name"
  value {
    s: "z"
  }
}
attr {
  key: "dtype"
  value {
    type: DT_INT32
  }
}
attr {
  key: "shape"
  value {
    shape {
      dim {
        size: -1
      }
    }
  }
}



In [20]:
print(list(pow_op.inputs))
print('-'*50)
print(list(pow_op.outputs))


[]
--------------------------------------------------
[<tf.Tensor 'z:0' shape=(None,) dtype=int32>]


In [21]:
#Placeholder用来放输入的地方，2.0中不需要，图中依然保留了
cube_func_int32.graph.get_operation_by_name("z")

<tf.Operation 'z' type=Placeholder>

In [22]:
cube_func_int32.graph.get_tensor_by_name("z:0")

<tf.Tensor 'z:0' shape=(None,) dtype=int32>

In [23]:
#打印出来看看图信息
cube_func_int32.graph.as_graph_def()

node {
  name: "z"
  op: "Placeholder"
  attr {
    key: "_user_specified_name"
    value {
      s: "z"
    }
  }
  attr {
    key: "dtype"
    value {
      type: DT_INT32
    }
  }
  attr {
    key: "shape"
    value {
      shape {
        dim {
          size: -1
        }
      }
    }
  }
}
node {
  name: "Pow/y"
  op: "Const"
  attr {
    key: "dtype"
    value {
      type: DT_INT32
    }
  }
  attr {
    key: "value"
    value {
      tensor {
        dtype: DT_INT32
        tensor_shape {
        }
        int_val: 3
      }
    }
  }
}
node {
  name: "Pow"
  op: "Pow"
  input: "z"
  input: "Pow/y"
  attr {
    key: "T"
    value {
      type: DT_INT32
    }
  }
}
node {
  name: "Identity"
  op: "Identity"
  input: "Pow"
  attr {
    key: "T"
    value {
      type: DT_INT32
    }
  }
}
versions {
  producer: 1994
}